In [ ]:
import numpy as np
from scipy import signal
from scipy.interpolate import CubicSpline
from scipy.signal import butter, filtfilt, find_peaks
from pathlib import Path

def butter_lowpass_filter(data, cutoff, fs, order=3):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

## Bandpass
def butter_bandpass_filter(data, lowcut, highcut, fs, order=3):
    nyquist = 0.5 * fs
    # For bandpass, the critical frequencies are normalized as a list [low, high]
    low = lowcut / nyquist
    high = highcut / nyquist

    # btype is changed to 'band'
    b, a = butter(order, [low, high], btype='band', analog=False)

    # filtfilt applies the filter forward and backward for zero phase-shift
    y = filtfilt(b, a, data)
    return y

def detect_valleys_ppg(signal, fs):
    """Detects valleys (feet) of PPG pulses."""
    inverted_signal = -signal
    valleys, _ = find_peaks(inverted_signal, distance=fs*0.2, prominence=0.1)
    if len(valleys) < 2:
        return np.array([0, len(signal)-1])
    return valleys

def spline_baseline_remove(signal, fs):
    """Removes baseline drift using Cubic Spline on valleys."""
    valleys = detect_valleys_ppg(signal, fs)
    xk = valleys
    yk = signal[valleys]
    try:
        cs = CubicSpline(xk, yk, bc_type='natural')
        baseline = cs(np.arange(len(signal)))
        return signal - baseline
    except:
        return signal - np.mean(signal)